# original-eat-rest-overcrowding-v4 -- real simulation test, seeds 1-10

Runs the `original-eat-rest-overcrowding-v4` external candidate through
`external/candidates/run_candidate.py` -- the same headless game loop as
`training/batch_runner.py`'s `_run_one`: `SimulationCore(seed=seed, starting_predators=0)`,
`MAX_SIM_TIME=3000`, one `sim.step()` per tick, candidate's own `make_policy()`/`decide_all()`
called every step. This is a real full game to extinction or the 3000s time limit each run,
not the candidate's own decision-check suite (its `validation.json` ships
`full_simulations_run: 0`).

This notebook actually **executes** the candidate rather than loading a precomputed CSV.
The `original-eat-rest-overcrowding-v4.zip` at the repo root was verified byte-for-byte
identical to the already-extracted `external/candidates/original-eat-rest-overcrowding-v4/`,
so no re-extraction step is needed.

**Cluster setup:** the first cells clone/refresh the repo the same way
`survival_baseline.ipynb` does. Before running on the cluster, create a git-ignored `.env`
file in the kernel's starting directory containing `GITHUB_TOKEN=<a token with repo read access>`.

Extends the existing seeds 1-3 result (`results/EXTERNAL_original-eat-rest-overcrowding-v4_seeds1-3.csv`)
to seeds 1-10.


In [4]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}


/home/jovyan/Nordic-AI-cup-2026 already cloned - skipping (use `git pull` there to update)


In [5]:
import glob
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

# Must run from survival-simulator/ so `src`, `agents`, `training` import.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])

print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)


From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
M	Nordic-AI-Cup-2026-main/survival-simulator/results/index.csv
Already on 'challenge-1V2'
Your branch is up to date with 'origin/challenge-1V2'.
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
Already up to date.
cwd: /home/jovyan/Nordic-AI-cup-2026/Nordic-AI-Cup-2026-main/survival-simulator
8e074cd whole lotta shit



In [6]:
!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

  Using cached fastapi-0.121.2-py3-none-any.whl.metadata (28 kB)
  Using cached numpy-2.3.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached pydantic-2.12.4-py3-none-any.whl.metadata (89 kB)
  Using cached pygame-2.6.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached scipy-1.16.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (62 kB)
  Using cached shapely-2.1.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.8 kB)
  Using cached uvicorn-0.38.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached pytest-9.1.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached imageio_ffmpeg-0.6.0-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached starlette-0.49.3-py3-none-any.whl.metadata (6.4 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
  Using cached pydantic_core-2.41

In [7]:
import subprocess
import sys
import time

SEEDS = list(range(1, 11))
SCRIPT = "external/candidates/run_candidate.py"
OUT_CSV = "results/EXTERNAL_original-eat-rest-overcrowding-v4_seeds1-10.csv"

cmd = [sys.executable, SCRIPT, "--seeds", *map(str, SEEDS), "--out", OUT_CSV]
print("running:", " ".join(cmd))

t0 = time.perf_counter()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
elapsed = time.perf_counter() - t0

print(f"\nfinished in {elapsed / 60:.1f} min, exit code {proc.returncode}")
if proc.returncode != 0:
    raise RuntimeError(f"run_candidate.py failed with exit code {proc.returncode}")


running: /opt/conda/bin/python external/candidates/run_candidate.py --seeds 1 2 3 4 5 6 7 8 9 10 --out results/EXTERNAL_original-eat-rest-overcrowding-v4_seeds1-10.csv
/opt/conda/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/opt/conda/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/opt/conda/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated

In [8]:
import pandas as pd

df = pd.read_csv(OUT_CSV)
df.loc[df["agent"] == "original_eat_rest_overcrowding_v4", "agent"] = "v4"
df = df.sort_values("seed").reset_index(drop=True)
df


,agent,seed,score,extinction_time,end_reason,wall_clock_sec
0,v4,1,1904.8107,1887.8,extinction,169.63
1,v4,2,1621.3882,1618.9,extinction,186.52
2,v4,3,1830.6998,1831.2,extinction,199.07
3,v4,4,791.1374,833.2,extinction,67.92
4,v4,5,1268.1955,1242.1,extinction,126.05
5,v4,6,1643.8758,1589.9,extinction,137.47
6,v4,7,1837.6815,1808.9,extinction,181.82
7,v4,8,1748.5609,1726.5,extinction,172.66
8,v4,9,1290.9453,1272.1,extinction,92.70
9,v4,10,1573.9488,1579.3,extinction,127.18


In [9]:
df["score"].agg(["mean", "median", "std", "min", "max"])


mean      1551.124390
median    1632.632000
std        343.791844
min        791.137400
max       1904.810700
Name: score, dtype: float64